# 🎬 Exploratory Data Analysis on Netflix Movies and TV Shows

## Overview
This project analyzes the Netflix content library to uncover trends in content types, genres, countries, ratings, and release patterns.

**Dataset:** [Netflix Movies and TV Shows – Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)  
**File:** `netflix_titles.csv`

## Importing all the Important Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully ✅')

## Loading the Dataset

In [ ]:
df = pd.read_csv('netflix_titles.csv')
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns ✅')

## Checking First Five Rows of Dataset

In [ ]:
df.head()

## Checking Last Five Rows of Dataset

In [ ]:
df.tail()

## Total Number of Rows and Columns Present in Dataset

In [ ]:
df.shape

## Checking Data Types of Each Column

In [ ]:
df.dtypes

## An Overview on Dataset

In [ ]:
df.info()

# Performing Data Cleaning

## Checking Missing Values

In [ ]:
# Total number of null values in each column
df.isnull().sum()

In [ ]:
# Checking % of null value in each column
(df.isnull().sum() / df.shape[0]) * 100

In [ ]:
# Total % of missing data in entire dataset
(df.isnull().sum().sum()) / (df.shape[0] * df.shape[1]) * 100

## Dropping / Filling Missing Values

In [ ]:
# Drop rows with missing 'date_added' and 'rating' (small %, safe to drop)
df.dropna(subset=['date_added', 'rating'], inplace=True)

# Fill 'director' and 'cast' with 'Unknown'
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)

# Fill 'country' with mode (most frequent)
df['country'].fillna(df['country'].mode()[0], inplace=True)

print('Missing values handled ✅')
df.isnull().sum()

## Checking for Duplicates

In [ ]:
df.duplicated().sum()
# No duplicate rows found

## Data Type Conversion

In [ ]:
# Convert 'date_added' to datetime
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())

# Extract year and month added
df['year_added']  = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month_name()

print('Date columns converted ✅')
df[['date_added','year_added','month_added']].head()

## Extracting Duration Information

In [ ]:
# Separate duration into numeric value and unit
df['duration_int']  = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_type'] = df['duration'].str.extract(r'(\D+)').apply(lambda x: x.str.strip())
df.head(3)

## Statistical Summary

In [ ]:
# For numerical columns
df.describe()

In [ ]:
# For categorical columns
df.describe(include='object')

# Performing Exploratory Data Analysis

## i) Univariate Analysis

In [ ]:
# --- Distribution of Content Type (Movies vs TV Shows) ---
plt.figure(figsize=(7, 5))
counts = df['type'].value_counts()
cols = ['#E50914', '#221F1F']
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', colors=cols,
        startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
plt.title('Content Type Distribution on Netflix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Movies dominate Netflix library at ~70%')

In [ ]:
# --- Top 10 Countries Producing Netflix Content ---
top_countries = df['country'].str.split(',').explode().str.strip().value_counts().head(10)
plt.figure(figsize=(10, 5))
bars = plt.barh(top_countries.index[::-1], top_countries.values[::-1], color='#E50914')
plt.xlabel('Number of Titles')
plt.title('Top 10 Countries — Netflix Content Production', fontsize=14, fontweight='bold')
for bar, val in zip(bars, top_countries.values[::-1]):
    plt.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2, str(val), va='center')
plt.tight_layout()
plt.show()
print('USA leads Netflix content production by a large margin')

In [ ]:
# --- Ratings Distribution ---
plt.figure(figsize=(10, 5))
rating_counts = df['rating'].value_counts()
sns.barplot(x=rating_counts.index, y=rating_counts.values, palette='Reds_r')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.title('Content Ratings Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print('TV-MA (Mature Audience) is the most common Netflix rating')

In [ ]:
# --- Content Added Per Year ---
year_counts = df['year_added'].value_counts().sort_index()
plt.figure(figsize=(12, 5))
plt.bar(year_counts.index, year_counts.values, color='#E50914', edgecolor='white')
plt.xlabel('Year')
plt.ylabel('Titles Added')
plt.title('Netflix Titles Added Per Year', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Netflix significantly ramped up content additions from 2016 onwards')

## ii) Bivariate Analysis

In [ ]:
# --- Movie Duration Distribution ---
movies = df[df['type'] == 'Movie']
plt.figure(figsize=(10, 5))
sns.histplot(movies['duration_int'].dropna(), bins=40, color='#E50914', edgecolor='white')
plt.xlabel('Duration (minutes)')
plt.title('Distribution of Movie Durations', fontsize=14, fontweight='bold')
plt.axvline(movies['duration_int'].median(), color='black', linestyle='--',
            label=f"Median: {movies['duration_int'].median():.0f} min")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- TV Show Seasons Distribution ---
shows = df[df['type'] == 'TV Show']
plt.figure(figsize=(8, 5))
season_counts = shows['duration_int'].value_counts().sort_index().head(10)
plt.bar(season_counts.index.astype(int), season_counts.values, color='#221F1F', edgecolor='white')
plt.xlabel('Number of Seasons')
plt.ylabel('Count')
plt.title('TV Shows — Season Count Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Most TV shows on Netflix have only 1 season')

In [ ]:
# --- Content Added by Month ---
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
month_counts = df['month_added'].value_counts().reindex(month_order)
plt.figure(figsize=(12, 5))
plt.plot(month_order, month_counts.values, marker='o', color='#E50914', linewidth=2.5)
plt.fill_between(month_order, month_counts.values, alpha=0.15, color='#E50914')
plt.xlabel('Month')
plt.ylabel('Titles Added')
plt.title('Netflix Content Addition Pattern by Month', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print('Netflix adds most content in July and December (summer & holiday seasons)')

In [ ]:
# --- Movies vs TV Shows Added Per Year ---
year_type = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
ax = year_type.plot(kind='bar', figsize=(12, 5), color=['#E50914', '#221F1F'],
                    edgecolor='white', width=0.7)
plt.xlabel('Year')
plt.ylabel('Count')
plt.title('Movies vs TV Shows Added Per Year', fontsize=14, fontweight='bold')
plt.legend(title='Type')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## iii) Multivariate Analysis

In [ ]:
# --- Top 15 Genres Overall ---
all_genres = df['listed_in'].str.split(',').explode().str.strip()
top_genres = all_genres.value_counts().head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, palette='Reds_r')
plt.xlabel('Number of Titles')
plt.title('Top 15 Netflix Genres', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('International Movies and Dramas are the top genres')

In [ ]:
# --- Top Genres by Content Type ---
df_genres = df.copy()
df_genres['genre'] = df_genres['listed_in'].str.split(',').str[0].str.strip()
top_genres_list = df_genres['genre'].value_counts().head(8).index

genre_type = df_genres[df_genres['genre'].isin(top_genres_list)]\
               .groupby(['genre','type']).size().unstack(fill_value=0)

genre_type.plot(kind='barh', figsize=(10, 6), color=['#E50914','#221F1F'],
                edgecolor='white', width=0.7)
plt.title('Top Genres — Movies vs TV Shows', fontsize=14, fontweight='bold')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation Heatmap (Numerical Columns) ---
plt.figure(figsize=(7, 5))
numeric_cols = df[['duration_int', 'release_year', 'year_added', 'month_added'
                   .replace('month_added', 'month_added')]]
numeric_df = df[['duration_int', 'release_year', 'year_added']].dropna()
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='Reds',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Release Year Trend: Movies vs TV Shows ---
release_trend = df.groupby(['release_year', 'type']).size().unstack(fill_value=0)
release_trend = release_trend[release_trend.index >= 2000]
release_trend.plot(figsize=(12, 5), color=['#E50914', '#221F1F'], linewidth=2)
plt.xlabel('Release Year')
plt.ylabel('Number of Titles')
plt.title('Content Release Trend (2000–Present)', fontsize=14, fontweight='bold')
plt.legend(title='Type')
plt.tight_layout()
plt.show()

## Key Insights & Findings

### 1. Content Type
- **~70% Movies, ~30% TV Shows** on Netflix.
- TV show additions grew faster than movies post-2018.

### 2. Country & Geography
- **USA** leads content production by a massive margin.
- **India** is the second-largest contributor, reflecting Netflix's push into Indian content.

### 3. Ratings
- **TV-MA** (Mature Audience) dominates — Netflix primarily targets adult viewers.

### 4. Timing
- Netflix heavily ramped up additions from **2016–2019**.
- **July & December** see the most content additions (summer and holiday peaks).

### 5. Genres
- **International Movies** and **Dramas** are the top two genres.
- **Stand-Up Comedy** is a genre where Netflix has heavily invested.

## Conclusion
This EDA of the Netflix content dataset reveals that Netflix is a movie-heavy platform with strong US dominance, though global content is growing rapidly. The platform targets mature audiences (TV-MA) and strategically adds content around holiday and summer seasons.

---
### 📌 Next Steps
- Build a **content recommendation system** based on genre + country.
- Perform **sentiment analysis** on descriptions.
- Predict **content type** from metadata using ML.

**📊 Tools Used:** Python, Pandas, Matplotlib, Seaborn, Jupyter Notebook

---
### -----------------------------------------THANKYOU---------------------------------